In [0]:
CATALOG = "workspace"

BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
METADATA_SCHEMA = "metadata"
AUDIT_SCHEMA = "audit"

RAW_DATA_PATH = "/Volumes/workspace/default/raw_data"

In [0]:
from pyspark.sql.types import *

audit_schema = StructType([
    StructField("batch_id", StringType(), False),
    StructField("pipeline_name", StringType(), False),
    StructField("table_name", StringType(), False),
    StructField("start_time", TimestampType(), False),
    StructField("end_time", TimestampType(), True),
    StructField("records_loaded", IntegerType(), True),
    StructField("status", StringType(), False),
    StructField("error_message", StringType(), True)
])

In [0]:
import uuid
from datetime import datetime

PIPELINE_NAME = "Bronze Ingestion"
BATCH_ID = str(uuid.uuid4())

In [0]:
metadata_df = spark.table(f"{CATALOG}.{METADATA_SCHEMA}.config")

display(metadata_df)

source_name,file_name,target_table,primary_key,active_flag
patients,patients.csv,patients,patient_id,Y
doctors,doctors.csv,doctors,doctor_id,Y
appointments,appointments.csv,appointments,appointment_id,Y
treatments,treatments.csv,treatments,treatment_id,Y
billing,billing.csv,billing,bill_id,Y


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
def load_to_bronze(file_name, table_name):

    file_path = f"{RAW_DATA_PATH}/{file_name}"

    print(f"Reading {file_path}")

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(file_path)
    )

    df = (
        df
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file", lit(file_name))
        .withColumn("load_date", current_date())
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}")
    )

    print(f"Loaded -> {table_name}")

In [0]:
from pyspark.sql.functions import *

def load_to_bronze(file_name, table_name):

    start_time = datetime.now()

    try:

        file_path = f"{RAW_DATA_PATH}/{file_name}"

        print(f"Loading {file_name}")

        df = (
            spark.read
                 .option("header", True)
                 .option("inferSchema", True)
                 .csv(file_path)
        )

        record_count = df.count()

        df = (
            df
            .withColumn("batch_id", lit(BATCH_ID))
            .withColumn("ingestion_timestamp", current_timestamp())
            .withColumn("source_file", lit(file_name))
            .withColumn("load_date", current_date())
        )

        (
            df.write
              .format("delta")
              .mode("overwrite")
              .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}")
        )

        audit = [
            (
                BATCH_ID,
                PIPELINE_NAME,
                table_name,
                start_time,
                datetime.now(),
                record_count,
                "SUCCESS",
                None
            )
        ]

    except Exception as e:

        audit = [
            (
                BATCH_ID,
                PIPELINE_NAME,
                table_name,
                start_time,
                datetime.now(),
                0,
                "FAILED",
                str(e)
            )
        ]

    audit_df = spark.createDataFrame(audit, audit_schema)

    (
        audit_df.write
                .mode("append")
                .saveAsTable("workspace.audit.pipeline_log")
    )

In [0]:
display(
    spark.table("workspace.audit.pipeline_log")
)

batch_id pipeline_name table_name start_time end_time records_loaded status error_message 226063ca-f8e5-4f28-844a-9d116b18e8a4 Bronze Ingestion patients 2026-07-02T06:49:15.501Z 2026-07-02T06:49:17.963Z 0 FAILED [DELTA_METADATA_MISMATCH] A metadata mismatch was detected when writing to the Delta table.
- A schema mismatch detected when writing to the Delta table (Table ID: 417df6d1-9612-4c8e-b037-908cf669dd44).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set: '.option("mergeSchema", "true")'.
For other operations, set the session configuration spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation specific to the operation for details.

Table schema:
root
 |-- patient_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- contact_number: long (nullable = true)
 |-- address: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- insurance_provider: string (nullable = true)
 |-- insurance_number: string (nullable = true)
 |-- email: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- load_date: date (nullable = true)


Data schema:
root
 |-- patient_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- contact_number: long (nullable = true)
 |-- address: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- insurance_provider: string (nullable = true)
 |-- insurance_number: string (nullable = true)
 |-- email: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- load_date: date (nullable = true)


- To overwrite your schema or change partitioning, please set: '.option("overwriteSchema", "true")'.
Note that the schema can't be overwritten when using 'replaceWhere'.
- Table ACLs are enabled in this cluster, so automatic schema migration is not allowed. Please use the ALTER TABLE command for changing the schema.

JVM stacktrace:
com.databricks.sql.transaction.tahoe.DeltaAnalysisExceptionWithSubErrors
	at com.databricks.sql.transaction.tahoe.MetadataMismatchErrorBuilder.finalizeAndThrow(DeltaErrors.scala:4603)
	at com.databricks.sql.transaction.tahoe.schema.ImplicitMetadataOperation.updateMetadata(ImplicitMetadataOperation.scala:238)
	at com.databricks.sql.transaction.tahoe.schema.ImplicitMetadataOperation.updateMetadata$(ImplicitMetadataOperation.scala:92)
	at com.databricks.sql.transaction.tahoe.commands.WriteIntoDeltaEdge.updateMetadata(WriteIntoDeltaEdge.scala:135)
	at com.databricks.sql.transaction.tahoe.commands.WriteIntoDeltaEdge.writeAndReturnCommitDataAndMaterializationPlans(WriteIntoDeltaEdge.scala:498)
	at com.databricks.sql.transaction.tahoe.commands.WriteIntoDeltaEdge.writeAndReturnCommitData(WriteIntoDeltaEdge.scala:334)
	at com.databricks.sql.transaction.tahoe.commands.CreateDeltaTableCommand.doDeltaWrite$1(CreateDeltaTableCommand.scala:724)
	at com.databricks.sql.transaction.tahoe.commands.CreateDeltaTableCommand.handleCreateTableAsSelect(CreateDeltaTableCommand.scala:874)
	at com.databricks.sql.transaction.tahoe.commands.CreateDeltaTableCommand.$anonfun$handleCommit$1(CreateDeltaTableCommand.scala:459)
	at com.databricks.sql.transaction.tahoe.OptimisticTransaction$.withActive(OptimisticTransaction.scala:289)
	at com.databricks.sql.transaction.tahoe.commands.CreateDeltaTableCommand.handleCommit(CreateDeltaTableCommand.scala:439)
	at com.databricks.sql.transaction.tahoe.commands.CreateDeltaTableCommand.$anonfun$run$8(CreateDeltaTableCommand.scala:342)
	at com.databricks.sql.transaction.tahoe.metering.DeltaLogging.withOperationTypeTag(DeltaLogging.sc

In [0]:
metadata = metadata_df.collect()

for row in metadata:

    if row.active_flag == "Y":

        load_to_bronze(
            row.file_name,
            row.target_table
        )

Loading patients.csv
Loading doctors.csv
Loading appointments.csv
Loading treatments.csv
Loading billing.csv


In [0]:
spark.sql(f"SHOW TABLES IN {CATALOG}.{BRONZE_SCHEMA}").show(truncate=False)

+--------+------------+-----------+
|database|tableName   |isTemporary|
+--------+------------+-----------+
|bronze  |appointments|false      |
|bronze  |billing     |false      |
|bronze  |doctors     |false      |
|bronze  |patients    |false      |
|bronze  |treatments  |false      |
+--------+------------+-----------+



In [0]:
display(
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.patients")
)

patient_id,first_name,last_name,gender,date_of_birth,contact_number,address,registration_date,insurance_provider,insurance_number,email,ingestion_timestamp,source_file,load_date
P001,David,Williams,F,1955-06-04,6939585183,789 Pine Rd,2022-06-23,WellnessCorp,INS840674,david.williams@mail.com,2026-07-02T06:36:35.908Z,patients.csv,2026-07-02
P002,Emily,Smith,F,1984-10-12,8228188767,321 Maple Dr,2022-01-15,PulseSecure,INS354079,emily.smith@mail.com,2026-07-02T06:36:35.908Z,patients.csv,2026-07-02
P003,Laura,Jones,M,1977-08-21,8397029847,321 Maple Dr,2022-02-07,PulseSecure,INS650929,laura.jones@mail.com,2026-07-02T06:36:35.908Z,patients.csv,2026-07-02
P004,Michael,Johnson,F,1981-02-20,9019443432,123 Elm St,2021-03-02,HealthIndia,INS789944,michael.johnson@mail.com,2026-07-02T06:36:35.908Z,patients.csv,2026-07-02
P005,David,Wilson,M,1960-06-23,7734463155,123 Elm St,2021-09-29,MedCare Plus,INS788105,david.wilson@mail.com,2026-07-02T06:36:35.908Z,patients.csv,2026-07-02
P006,Linda,Jones,M,1963-06-16,7561777264,321 Maple Dr,2022-10-02,HealthIndia,INS613758,linda.jones@mail.com,2026-07-02T06:36:35.908Z,patients.csv,2026-07-02
P007,Alex,Johnson,F,1989-06-08,6278710077,789 Pine Rd,2021-12-25,MedCare Plus,INS465890,alex.johnson@mail.com,2026-07-02T06:36:35.908Z,patients.csv,2026-07-02
P008,David,Davis,F,1976-07-05,7090558393,456 Oak Ave,2021-05-25,WellnessCorp,INS545101,david.davis@mail.com,2026-07-02T06:36:35.908Z,patients.csv,2026-07-02
P009,Laura,Davis,M,1971-12-11,7060324619,321 Maple Dr,2022-09-18,PulseSecure,INS136631,laura.davis@mail.com,2026-07-02T06:36:35.908Z,patients.csv,2026-07-02
P010,Michael,Taylor,M,2001-10-13,7081396733,123 Elm St,2022-08-24,WellnessCorp,INS866577,michael.taylor@mail.com,2026-07-02T06:36:35.908Z,patients.csv,2026-07-02


Bronze is the raw landing zone.
It preserves the original data.
If a transformation goes wrong later, you can always rebuild Silver and Gold from Bronze without asking the source system to resend the data.

In [0]:
from pyspark.sql.functions import current_timestamp
import uuid

BATCH_ID = str(uuid.uuid4())

print(BATCH_ID)